<a href="https://colab.research.google.com/github/Luigi2908/Segmentaci-n-e-commerce/blob/main/Proyecto_3_Customer_Segmentation_with_RFM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
datacertlaboratoria_proyecto_3_segmentacin_de_clientes_en_ecommerce_path = kagglehub.dataset_download('datacertlaboratoria/proyecto-3-segmentacin-de-clientes-en-ecommerce')

print('Data source import complete.')


Using Colab cache for faster access to the 'proyecto-3-segmentacin-de-clientes-en-ecommerce' dataset.
Data source import complete.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Customer Segmentation with RFM ( Proyecto 3: Segmentación de Clientes en ecommerce)

import numpy as np
import pandas as pd
test = pd.read_csv("/ventas-por-factura.csv")
df = test.copy()
df.head()

,N° de factura,Fecha de factura,ID Cliente,País,Cantidad,Monto
0,548370,3/30/2021 16:14:00,15528.0,United Kingdom,123,"229,33"
1,575767,11/11/2021 11:11:00,17348.0,United Kingdom,163,"209,73"
2,C570727,10/12/2021 11:32:00,12471.0,Germany,-1,"-1,45"
3,549106,4/6/2021 12:08:00,17045.0,United Kingdom,1,"39,95"
4,573112,10/27/2021 15:33:00,16416.0,United Kingdom,357,"344,83"


In [ ]:
# N° de factura    : Invoice_ID
# Fecha de factura : Invoice_Date
# ID Cliente       : Customer_ID
# País             : Country
# Cantidad         : Quantity
# Monto            : Amount


df.columns = ["Invoice_ID","Invoice_Date", "Customer_ID","Country",
            "Quantity","Amount"]
df.head()



,Invoice_ID,Invoice_Date,Customer_ID,Country,Quantity,Amount
0,548370,3/30/2021 16:14:00,15528.0,United Kingdom,123,"229,33"
1,575767,11/11/2021 11:11:00,17348.0,United Kingdom,163,"209,73"
2,C570727,10/12/2021 11:32:00,12471.0,Germany,-1,"-1,45"
3,549106,4/6/2021 12:08:00,17045.0,United Kingdom,1,"39,95"
4,573112,10/27/2021 15:33:00,16416.0,United Kingdom,357,"344,83"


In [ ]:
df.info()


# separate the date variable from time
df["Invoice_Date"] = pd.to_datetime(df["Invoice_Date"]).dt.date
# convert to datetime
df["Invoice_Date"] = pd.to_datetime(df["Invoice_Date"])

# # convert object to float

df["Amount"] = df["Amount"].str.replace(",",".").astype(float)


df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25953 entries, 0 to 25952
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Invoice_ID    25953 non-null  object 
 1   Invoice_Date  25953 non-null  object 
 2   Customer_ID   22229 non-null  float64
 3   Country       25953 non-null  object 
 4   Quantity      25953 non-null  int64  
 5   Amount        25953 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 1.2+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25953 entries, 0 to 25952
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Invoice_ID    25953 non-null  object        
 1   Invoice_Date  25953 non-null  datetime64[ns]
 2   Customer_ID   22229 non-null  float64       
 3   Country       25953 non-null  object        
 4   Quantity      25953 non-null  int64         
 5   Amount        25953 non

In [ ]:
df.isnull().sum()
df=df.dropna()
df.isnull().sum()

,0
Invoice_ID,0
Invoice_Date,0
Customer_ID,0
Country,0
Quantity,0
Amount,0


In [ ]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Invoice_Date,22229,2021-06-29 04:40:49.340951040,2020-12-01 00:00:00,2021-03-28 00:00:00,2021-07-08 00:00:00,2021-10-10 00:00:00,2021-12-09 00:00:00,NaN
Customer_ID,22229.0,15238.289892,12346.0,13755.0,15136.0,16746.0,18287.0,1732.981685
Quantity,22229.0,220.766026,-80995.0,30.0,120.0,254.0,80995.0,1169.102426
Amount,22229.0,373.465218,-168469.6,87.3,240.68,417.69,168469.6,2016.647262


In [ ]:
#take out the return invoice from df   ( # C570727)

df.head()
df = df[~df["Invoice_ID"].str.contains("C")]

df.describe().T

,count,mean,min,25%,50%,75%,max,std
Invoice_Date,18570,2021-07-01 01:31:25.492730112,2020-12-01 00:00:00,2021-03-30 00:00:00,2021-07-12 00:00:00,2021-10-12 00:00:00,2021-12-09 00:00:00,NaN
Customer_ID,18570.0,15265.970813,12346.0,13772.5,15173.0,16779.0,18287.0,1734.016581
Quantity,18570.0,279.071459,1.0,74.0,155.0,290.75,80995.0,975.795133
Amount,18570.0,479.977379,0.0,157.685,302.865,471.2575,168469.6,1676.431595


In [ ]:
df.head()
df[df["Amount"] == 0.0].shape

# take out the Amount zero values from df
df = df[df["Amount"] != 0.0]

In [ ]:
df.head()
df.groupby("Customer_ID")["Invoice_ID"].count()



,Invoice_ID
Customer_ID,
12346.0,1
12347.0,7
12348.0,4
12349.0,1
12350.0,1
...,...
18280.0,1
18281.0,1
18282.0,2


In [ ]:
df.head()
import datetime as dt
# Preparation of RFM metrics

df["Invoice_Date"].max()   # 2021-12-09

analysis_date = dt.datetime(2021,12, 10)

df_rfm = df.groupby("Customer_ID").agg({"Quantity": lambda x : x.sum(),
                               "Amount" : lambda x : x.sum(),
                               "Invoice_ID": "count",
                               "Invoice_Date" : lambda x: (analysis_date - x.max()).days})
df_rfm.head()

,Quantity,Amount,Invoice_ID,Invoice_Date
Customer_ID,,,,
12346.0,74215,77183.60,1,326
12347.0,2458,4310.00,7,3
12348.0,2341,1797.24,4,76
12349.0,631,1757.55,1,19
12350.0,197,334.40,1,311


In [ ]:
rfm = df_rfm
rfm.columns = ["Quantity","Monetary","Frequency","Recency"]

rfm = rfm[rfm["Frequency"] > 1 ]
rfm.head()


,Quantity,Monetary,Frequency,Recency
Customer_ID,,,,
12347.0,2458,4310.00,7,3
12348.0,2341,1797.24,4,76
12352.0,536,2506.04,8,37
12356.0,1591,2811.43,3,23
12358.0,248,1168.06,2,2


In [ ]:
# calculation of rfm scores


rfm["Recency_score"]= pd.qcut(rfm["Recency"], 5, labels = [5, 4, 3, 2, 1])
rfm["Frequency_score"]= pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
rfm["Monetary_score"]= pd.qcut(rfm["Monetary"], 5, labels = [1, 2, 3, 4, 5])

rfm["RFM_SCORE"] = (rfm['Recency_score'].astype(str) + rfm['Frequency_score'].astype(str))

rfm.head()



/tmp/ipykernel_15135/3081911322.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rfm["Recency_score"]= pd.qcut(rfm["Recency"], 5, labels = [5, 4, 3, 2, 1])
/tmp/ipykernel_15135/3081911322.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rfm["Frequency_score"]= pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
/tmp/ipykernel_15135/3081911322.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] =

,Quantity,Monetary,Frequency,Recency,Recency_score,Frequency_score,Monetary_score,RFM_SCORE
Customer_ID,,,,,,,,
12347.0,2458,4310.00,7,3,5,4,5,54
12348.0,2341,1797.24,4,76,2,3,4,23
12352.0,536,2506.04,8,37,3,5,4,35
12356.0,1591,2811.43,3,23,4,2,4,42
12358.0,248,1168.06,2,2,5,1,3,51


In [ ]:
# creating and analyzing RFM segments
seg_map = {
    r'[1-2][1-2]': 'hibernating',
    r'[1-2][3-4]': 'at_Risk',
    r'[1-2]5': 'cant_loose',
    r'3[1-2]': 'about_to_sleep',
    r'33': 'need_attention',
    r'[3-4][4-5]': 'loyal_customers',
    r'41': 'promising',
    r'51': 'new_customers',
    r'[4-5][2-3]': 'potential_loyalists',
    r'5[4-5]': 'champions'
}

rfm["segment"]= rfm["RFM_SCORE"].replace(seg_map, regex=True)

rfm.head()


/tmp/ipykernel_15135/1383153287.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rfm["segment"]= rfm["RFM_SCORE"].replace(seg_map, regex=True)


,Quantity,Monetary,Frequency,Recency,Recency_score,Frequency_score,Monetary_score,RFM_SCORE,segment
Customer_ID,,,,,,,,,
12347.0,2458,4310.00,7,3,5,4,5,54,champions
12348.0,2341,1797.24,4,76,2,3,4,23,at_Risk
12352.0,536,2506.04,8,37,3,5,4,35,loyal_customers
12356.0,1591,2811.43,3,23,4,2,4,42,potential_loyalists
12358.0,248,1168.06,2,2,5,1,3,51,new_customers


In [ ]:
# Analysis

rfm[["segment", "Recency","Frequency", "Monetary" ]].groupby("segment").agg(["mean", "count"])

Recency        Frequency           Monetary      
                           mean count       mean count         mean count
segment                                                                  
about_to_sleep        32.468085   188   2.324468   188   863.203032   188
at_Risk              104.819951   411   4.355231   411  1561.332311   411
cant_loose            95.745763    59  10.169492    59  3568.016102    59
champions              4.591260   389  15.688946   389  9000.500617   389
hibernating          137.054135   665   2.246617   665   949.549639   665
loyal_customers       22.623352   531   9.103578   531  4385.418211   531
need_attention        31.734513   113   3.548673   113  1304.096637   113
new_customers          5.750000    52   2.000000    52   746.871923    52
potential_loyalists   12.056657   353   3.226629   353  1635.912663   353
promising             16.476744    86   2.000000    86   675.962326    86

In [ ]:
rfm = rfm.reset_index()
champions_segments_customer = rfm[rfm["segment"].isin(["champions"])]["Customer_ID"]

custumors_champions = df[(df["Customer_ID"].isin(champions_segments_customer))]

custumors_champions.head(10)

,Invoice_ID,Invoice_Date,Customer_ID,Country,Quantity,Amount
6,538125,2020-12-09,18225.0,United Kingdom,16,30.00
9,570651,2021-10-11,14911.0,EIRE,86,321.35
14,580987,2021-12-06,17389.0,United Kingdom,96,720.00
27,576083,2021-11-14,17404.0,Sweden,1060,1238.48
31,552837,2021-05-11,12901.0,United Kingdom,72,140.04
48,565682,2021-09-06,14796.0,United Kingdom,467,835.95
55,579089,2021-11-28,17920.0,United Kingdom,58,111.11
57,571098,2021-10-13,17841.0,United Kingdom,200,437.43
58,574994,2021-11-08,17870.0,United Kingdom,62,143.11
59,570257,2021-10-10,13767.0,United Kingdom,129,380.31
